In [1]:
import sys
sys.path.append('..')
from pymavlink import mavutil
import matplotlib.pyplot as plt
import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point, LineString

from sqlalchemy import create_engine

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

In [2]:
# retorn conexion a base de datos
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# obtiene todos los recorridos de control biologico de la base de datos
def get_recorridos_de_bd():
    return gpd.read_postgis(
        "SELECT * FROM drones_control_bio.recorridos_lib", 
        obtener_engine(), 
        geom_col='geom'  # columna con la geometría
    )

# recibe el nombre de vehiculo y retorna ruta de sus logs
def get_ruta_logs_de_vehiculo(vehiculo):
    if vehiculo == 'DA01':
        return RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Trichogramma\2025\TLOGS\ALTA_01'
    elif vehiculo == 'DA02':
        return RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Trichogramma\2025\TLOGS\ALTA_02'
    elif vehiculo == 'DA03':
        return RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Trichogramma\2025\TLOGS\ALTA_03'
    elif vehiculo == 'DA04':
        return RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - EQUIPO AVIACION UTEA\Trichogramma\2025\TLOGS\ALTA_04'
    return ''

# recibe el path de logs y lista de logs
# conviente los logs en un GDF de lineas
def convertir_lista_logs_a_gdf(path_log_vehiculo, lista_logs_vehiculo):
    list_geo = []
    list_name = []
    for log_name in lista_logs_vehiculo:
        log = path_log_vehiculo + '/' + log_name
        mlog = mavutil.mavlink_connection(log)
        lats = []
        lons = []
        while True:
            msg = mlog.recv_match()
            if not msg:
                break
            # Ejemplo de procesamiento de mensaje
            if msg.get_type() == 'GLOBAL_POSITION_INT':
                lat = msg.lat / 1e7  # Latitud en grados
                lon = msg.lon / 1e7  # Longitud en grados
                # verifica si las coors no esten en el origen [0, 0]
                if lat == 0 and lon == 0:
                    continue
                lats.append(lat)
                lons.append(lon)
        # si la lista de latitudes y longitudes estan vacias, crea unos puntos arbitrarios
        # esto para no dejar recorridos vacios
        if len(lats) == 0 and len(lons) == 0:
            lats = [-17.2844492, -17.2844479, -17.2844473, -17.2844466]
            lons = [-63.1680544, -63.1680534, -63.1680534, -63.168053]
        coors = {'lat':lats, 'lon':lons}
        df = pd.DataFrame(coors)
        geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
        line = LineString(geometry)
        list_geo.append(line)
        list_name.append(log_name)
    gdf_lines = gpd.GeoDataFrame({'nombre': list_name, 'geometry': list_geo}, crs="EPSG:4326")
    return gdf_lines

# Función para contar la cantidad de coordenadas en una geometría de tipo LINESTRING
def get_cantidad_de_vertices(linea):
    return len(linea.coords)

# inserta los nuevos recorridos en la base de datos
def insert_nuevos_recorridos(gdf_recorridos_vehiculo):
    gdf_recorridos_vehiculo.to_postgis(
        name="recorridos_lib",
        schema="drones_control_bio",
        con=obtener_engine(),
        if_exists="append",   # usa "replace" si quieres sobrescribir todo
        index=False
    )

In [3]:
RUTA_COMPLETA = os.path.join(RUTA_UNIDAD_ONE_DRIVE, RUTA_LOCAL_ONE_DRIVE)

# lista de drones poara control bio
LISTA_VEHICULOS = ['DA01', 'DA02', 'DA03', 'DA04']
#LISTA_VEHICULOS = ['DA01']

# todos los recorridos
GDF_RECORRIDOS = get_recorridos_de_bd()
print(len(GDF_RECORRIDOS), 'recorridos')

2659 recorridos


In [4]:
for vehiculo in LISTA_VEHICULOS:
    # obtiene ruta de logs del vehiculos
    path_logs_de_vehiculo = get_ruta_logs_de_vehiculo(vehiculo)
    # verifuca si se encontro la ruta
    if path_logs_de_vehiculo == '':
        print(f'❌ {vehiculo} : Ruta de vehiculo no encontrado...!')
        continue
    # lista todos los logs que se encuantran en ruta
    lista_logs_de_vehiculo = os.listdir(path_logs_de_vehiculo)
    # filtra solo los recorridos que no estan en la base de datos
    logs_faltantes_de_vehiculo = [log_name for log_name in lista_logs_de_vehiculo if log_name not in GDF_RECORRIDOS['nombre'].values]
    # verifica si existen logs faltantes
    if len(logs_faltantes_de_vehiculo) == 0:
        print(f'⚠️ {vehiculo} : No existen logs para procesar.')
        continue
    
    #print(vehiculo)
    #print(path_logs_de_vehiculo)
    #print(logs_faltantes_de_vehiculo)
    
    # convertir lista de logs a GDF
    gdf_nuevos_recorridos = convertir_lista_logs_a_gdf(path_logs_de_vehiculo, logs_faltantes_de_vehiculo)
    # agregar campos faltante
    gdf_nuevos_recorridos['num_vert'] = gdf_nuevos_recorridos['geometry'].apply(get_cantidad_de_vertices)
    gdf_nuevos_recorridos['vehiculo'] = vehiculo
    gdf_nuevos_recorridos['idd'] = 0
    
    # reproyecta a UTM zona 20S
    gdf_nuevos_recorridos.crs = "EPSG:4326"
    gdf_nuevos_recorridos_utm = gdf_nuevos_recorridos.to_crs(epsg=32720)
    # reasigna la columna de geometria
    gdf_nuevos_recorridos_utm = gdf_nuevos_recorridos_utm.rename_geometry("geom")
    # registra nuevos recorridos en BD
    insert_nuevos_recorridos(gdf_nuevos_recorridos_utm)
    print(f'✅ {vehiculo} : Logs procesados con exito.')

⚠️ DA01 : No existen logs para procesar.
✅ DA02 : Logs procesados con exito.
⚠️ DA03 : No existen logs para procesar.
✅ DA04 : Logs procesados con exito.
